In [6]:
"""
Red Light Violation Detection - FIXED v2
"""
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort
import cv2, numpy as np, csv, os
from datetime import datetime
# ══════════════════════════════════════
VIDEO_PATH     = r"../Video_inpt/Red_Light_Violation_1.mp4"
OUTPUT_PATH    = r"../video_out/video_out_red_light_violation.avi"
VIOLATIONS_CSV = r"../VIOLATIONS_CSV/violation.csv"
VIOLATIONS_DIR = r"../VIOLATIONS_Frame/violation.jpg"
STOP_LINE      = [(420, 420), (940, 464)] 
# ══════════════════════════════════════

model   = YOLO(r"../Models/yolo11x_red_light.pt")
tracker = DeepSort(max_age=40, n_init=2)

os.makedirs(VIOLATIONS_DIR, exist_ok=True)
csv_file   = open(VIOLATIONS_CSV, "w", newline="", encoding="utf-8-sig")
csv_writer = csv.writer(csv_file)
csv_writer.writerow(["car_id", "frame", "timestamp", "image_path"])


def is_red_light(frame, boxes):
    """كشف الإشارة الحمراء بالـ HSV"""
    if not boxes:
        return False
    for (x1, y1, x2, y2) in boxes:
        roi = frame[y1:y2, x1:x2]
        if roi.size == 0:
            continue
        hsv  = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
        mask = cv2.inRange(hsv, np.array([0,120,70]),  np.array([10,255,255])) + \
               cv2.inRange(hsv, np.array([170,120,70]), np.array([180,255,255]))
        if cv2.countNonZero(mask) / (roi.shape[0]*roi.shape[1]+1e-5) > 0.05:
            return True
    return False


def get_side(px, py):
    """إيجابي = فوق الخط | سالب = تحت الخط"""
    (x1,y1),(x2,y2) = STOP_LINE
    return (x2-x1)*(py-y1) - (y2-y1)*(px-x1)


def draw_ui(frame, red, in_c, out_c, viol_c, fnum):
    h, w = frame.shape[:2]
    # cv2.line(frame, STOP_LINE[0], STOP_LINE[1], (255,0,255), 3)
    # cv2.putText(frame, "STOP LINE", (STOP_LINE[0][0], STOP_LINE[0][1]-8),
    #             cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,0,255), 2)
    # إشارة
    col = (0,0,255) if red else (0,200,0)
    cv2.rectangle(frame,(10,10),(250,48),col,-1)
    cv2.putText(frame,"Red Light: ON" if red else "Red Light: OFF",
                (15,36),cv2.FONT_HERSHEY_SIMPLEX,0.8,(255,255,255),2)
    # IN/OUT
    cv2.rectangle(frame,(10,58),(240,96),(30,30,30),-1)
    cv2.putText(frame,f"IN: {in_c} | OUT: {out_c}",(15,84),
                cv2.FONT_HERSHEY_SIMPLEX,0.8,(255,255,255),2)
    # مخالفات
    cv2.rectangle(frame,(10,106),(260,144),(0,0,180),-1)
    cv2.putText(frame,f"Violations: {viol_c}",(15,132),
                cv2.FONT_HERSHEY_SIMPLEX,0.8,(255,255,255),2)
    # فريم
    cv2.rectangle(frame,(w-100,10),(w-5,42),(0,0,0),-1)
    cv2.putText(frame,str(fnum),(w-95,34),cv2.FONT_HERSHEY_SIMPLEX,0.7,(255,255,255),2)
    cv2.putText(frame,datetime.now().strftime("%m/%d/%Y  %H:%M:%S"),
                (10,h-10),cv2.FONT_HERSHEY_SIMPLEX,0.6,(200,200,200),1)
    return frame


# ══════════════════════════════════════
cap = cv2.VideoCapture(VIDEO_PATH)
W   = int(cap.get(3)); H = int(cap.get(4))
FPS = cap.get(cv2.CAP_PROP_FPS) or 30
OUTPUT_PATH = r"C:\Users\Admin\Desktop\lectures\Capston_Project\Final_project\video_out\first_out.mp4"
out = cv2.VideoWriter(OUTPUT_PATH, cv2.VideoWriter_fourcc(*"XVID"), FPS, (W,H))

# ─── STATE ────────────────────────────────────────────────
prev_sides   = {}   # {tid: float}   ← الجانب في الفريم السابق
violated_ids = set()
in_count = out_count = frame_num = 0
# ──────────────────────────────────────────────────────────

print("🚀 Starting...")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    frame_num += 1

    # 1. إشارات المرور
    sig_res  = model(frame, classes=[9], verbose=False, conf=0.25)
    sig_boxes = [(int(b.xyxy[0][0]),int(b.xyxy[0][1]),
                  int(b.xyxy[0][2]),int(b.xyxy[0][3]))
                 for r in sig_res for b in r.boxes]
    for (x1,y1,x2,y2) in sig_boxes:
        cv2.rectangle(frame,(x1,y1),(x2,y2),(0,165,255),2)

    red_light = is_red_light(frame, sig_boxes)

    # 2. كشف السيارات
    car_res    = model(frame, classes=[2,3,5,7], verbose=False, conf=0.3)
    detections = [([int(b.xyxy[0][0]),int(b.xyxy[0][1]),
                    int(b.xyxy[0][2])-int(b.xyxy[0][0]),
                    int(b.xyxy[0][3])-int(b.xyxy[0][1])],
                   float(b.conf[0]), "car")
                  for r in car_res for b in r.boxes]

    # 3. تتبع
    tracks = tracker.update_tracks(detections, frame=frame)

    for track in tracks:
        if not track.is_confirmed(): continue

        tid            = track.track_id
        x1,y1,x2,y2   = map(int, track.to_ltrb())
        cx             = (x1+x2)//2
        cy             = y2          # أسفل السيارة

        curr_side = get_side(cx, cy)

        # ─── FIX: احفظ الجانب السابق قبل أي تعديل ───────
        old_side = prev_sides.get(tid, None)

        # عداد IN/OUT
        if old_side is not None:
            if old_side > 0 and curr_side <= 0:   # عدى من فوق لتحت
                out_count += 1
            elif old_side < 0 and curr_side >= 0: # عدى من تحت لفوق
                in_count  += 1

        # ─── كشف المخالفة ─────────────────────────────────
        # شرط المخالفة: إشارة حمراء + السيارة عدت الخط (من فوق لتحت)
        violated = tid in violated_ids

        if red_light and not violated and old_side is not None:
            if old_side > 0 and curr_side <= 0:
                violated_ids.add(tid)
                violated  = True

                # ─── رسم البوكس الأحمر فوراً قبل حفظ الفريم ───
                cv2.rectangle(frame,(x1,y1),(x2,y2),(0,0,255),2)
                cv2.rectangle(frame,(x1,y1-26),(x2,y1),(0,0,255),-1)
                cv2.putText(frame,f"car ID:{tid} Violated",(x1+4,y1-7),
                            cv2.FONT_HERSHEY_SIMPLEX,0.55,(255,255,255),2)

                now       = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                img_path  = os.path.join(VIOLATIONS_DIR, f"car{tid}_frame{frame_num}.jpg")
                cv2.imwrite(img_path, frame)
                csv_writer.writerow([tid, frame_num, now, img_path])
                csv_file.flush()
                print(f"🚨 Violation! Car ID:{tid} | Frame:{frame_num}")

        # تحديث الجانب بعد كل الحسابات
        prev_sides[tid] = curr_side

        # رسم (يعاد رسم نفس البوكس الأحمر للسيارات المخالفة فعلاً)
        color = (0,0,255) if violated else (0,255,0)
        label = f"car ID:{tid} Violated" if violated else f"car ID:{tid}"
        cv2.rectangle(frame,(x1,y1),(x2,y2),color,2)
        cv2.rectangle(frame,(x1,y1-26),(x2,y1),color,-1)
        cv2.putText(frame,label,(x1+4,y1-7),
                    cv2.FONT_HERSHEY_SIMPLEX,0.55,(255,255,255),2)

    frame = draw_ui(frame, red_light, in_count, out_count, len(violated_ids), frame_num)
    out.write(frame)

    if frame_num % 30 == 0:
        print(f"  Frame {frame_num:5d} | Red:{red_light} | Violations:{len(violated_ids)} | IN:{in_count} OUT:{out_count}")

cap.release(); out.release(); csv_file.close()
print(f"\n✅ Done! Violations: {len(violated_ids)} | Video: {OUTPUT_PATH}")

🚀 Starting...
  Frame    30 | Red:True | Violations:0 | IN:0 OUT:0
  Frame    60 | Red:True | Violations:0 | IN:1 OUT:0
  Frame    90 | Red:True | Violations:0 | IN:3 OUT:0
  Frame   120 | Red:True | Violations:0 | IN:3 OUT:0
  Frame   150 | Red:True | Violations:0 | IN:3 OUT:0
  Frame   180 | Red:True | Violations:0 | IN:3 OUT:0
🚨 Violation! Car ID:49 | Frame:189
  Frame   210 | Red:True | Violations:1 | IN:3 OUT:1
  Frame   240 | Red:True | Violations:1 | IN:3 OUT:1
  Frame   270 | Red:True | Violations:1 | IN:3 OUT:1
  Frame   300 | Red:True | Violations:1 | IN:3 OUT:1
  Frame   330 | Red:True | Violations:1 | IN:3 OUT:1
  Frame   360 | Red:True | Violations:1 | IN:3 OUT:1

✅ Done! Violations: 1 | Video: C:\Users\Admin\Desktop\lectures\Capston_Project\Final_project\video_out\first_out.mp4
